In [50]:
%pip install -qU google-generativeai langgraph tavily
%pip install -qU tavily-python
%pip install -qU ddgs

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [51]:
import os
import google.generativeai as genai
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [52]:
client = TavilyClient(api_key=TAVILY_API_KEY)
result = client.search("Que son multiagentes de Inteligencia Artificial?", 
                       include_answer=True)

result["answer"]

'Multiagentes de Inteligencia Artificial son sistemas de múltiples agentes colaborativos que trabajan juntos para resolver tareas complejas. Cada agente tiene una función especializada y se comunica para optimizar resultados. Esta colaboración mejora eficiencia y efectividad.'

In [53]:
ciudad = "Tulum"

query = f"""
Enumere los 5 principales restaurantes en {ciudad}, según evaluaciones recientes en TripAdvisor o sitios similares de turismo.
Para cada restaurante, indique:
- Tipo de cocina (ej.: regional, italiana, japonesa)
- Una breve descripción (máx. 2 líneas)
- Calificación promedio (si está disponible)
- Rango de precios

Responda únicamente con datos actualizados y relevantes para turistas.
"""

In [54]:
from ddgs import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        raise e

for link in search(query):
    print(link)

https://www.facebook.com/denise.owen.92/posts/villa-joya-sea-stands-on-xiringuito-beach-close-to-praia-da-galé-and-overlooking/10160367641943549/
https://costeno.com/
https://www.fiestamericanatravelty.com/devossion/hoteles/devossion-playa-del-carmen
https://www.puertoaventurashotel.com/
https://www.tripadvisor.com/Restaurants-g297478-zfp63-Medellin_Antioquia_Department.html
https://www.opentable.com/n/mexico/ciudad-de-mexico/santa-fe-bosques-restaurants


In [55]:
%pip install -qU bs4

Note: you may need to restart the kernel to use updated packages.


In [56]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

ddg = DDGS()

def scrape_restaurantes_info(url):
    if not url:
        print("Error: URL vacia o no localizada.")
        return None

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, como Gecko) Chrome/138.0.0.0 Safari/537.36',
        'Accept-Language': 'es-MX,es;q=0.9,en;q=0.8'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error al cargar la página {url}: {e}")
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

search_results = search(query)

if search_results:
    url = search_results[0]
else:
    url = None

soup = scrape_restaurantes_info(url)

print(f"Website: {url}\n\n")

if soup:
    print(str(soup.body.prettify())[:50000])
else:
    print("No fue posible raspar el contenido de la página o la URL no fue encontrada.")

Error al cargar la página https://www.tripadvisor.com.mx/Restaurants-g150813-Tulum_Yucatan_Peninsula.html: 403 Client Error: Forbidden for url: https://www.tripadvisor.com.mx/Restaurants-g150813-Tulum_Yucatan_Peninsula.html
Website: https://www.tripadvisor.com.mx/Restaurants-g150813-Tulum_Yucatan_Peninsula.html


No fue posible raspar el contenido de la página o la URL no fue encontrada.


In [57]:
import os
import re
from tavily import TavilyClient

api_key = os.environ.get("TAVILY_API_KEY")
if not api_key:
    raise ValueError("la clave TAVILY_API_KEY no fue encontrada.")

cliente_tavily = TavilyClient(api_key=api_key)

ciudad = "Tulum"
# Hacemos la consulta a Tavily más específica para traer listados de TripAdvisor
tavily_query = f"site:tripadvisor.com.mx/Restaurants restaurantes en {ciudad}"

print("Iniciando la búsqueda agéntica por URLs de Tripadvisor con Tavily...")
tripadvisor_url = None

try:
    # Aumentamos a 10 resultados para asegurar que encuentre TripAdvisor
    tavily_results = cliente_tavily.search(query=tavily_query, max_results=10)

    if tavily_results and tavily_results["results"]:
        print(f"Tavily encontró {len(tavily_results['results'])} resultados. Analizando...")
        for result in tavily_results["results"]:
            url = result["url"]
            # Validamos que sea un enlace de TripAdvisor y que sea un LISTADO (/Restaurants-)
            if ("tripadvisor.com" in url or "tripadvisor.com.mx" in url) and "/Restaurants-" in url:
                tripadvisor_url = url
                break
                
except Exception as e:
    print(f"Error en la búsqueda agéntica con Tavily: {e}.")

# RESPALDO: Si Tavily no encontró nada, asignamos la URL de listado general manualmente
if not tripadvisor_url:
    print("⚠️ Tavily no encontró un listado de TripAdvisor en esta ejecución. Asignando listado por defecto...")
    tripadvisor_url = "https://www.tripadvisor.com.mx/Restaurants-g150813-Tulum_Yucatan_Peninsula.html"

# Limpieza de paginación
if tripadvisor_url:
    clean_url = re.sub(r'-oa\d+-', '-', tripadvisor_url)
    tripadvisor_url = clean_url
    print(f"✅ URL encontrada limpia de paginación.")

print("*" * 50)
print(f"URL Final de Tripadvisor para el raspado: {tripadvisor_url}")
print("*" * 50)


Iniciando la búsqueda agéntica por URLs de Tripadvisor con Tavily...
Tavily encontró 10 resultados. Analizando...
✅ URL encontrada limpia de paginación.
**************************************************
URL Final de Tripadvisor para el raspado: https://www.tripadvisor.com.mx/Restaurants-g150813-Tulum_Yucatan_Peninsula.html
**************************************************


In [58]:
%pip install -qU selenium
%pip install -qU webdriver-manager

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [59]:
%pip install -qU setuptools undetected-chromedriver


Note: you may need to restart the kernel to use updated packages.


In [60]:
%pip install -qU undetected-chromedriver


Note: you may need to restart the kernel to use updated packages.


In [61]:
import sys
import types
import re

# --- PARCHE DE COMPATIBILIDAD PARA PYTHON 3.12+ (MOCK DE DISTUTILS) ---
try:
    from distutils.version import LooseVersion
except ModuleNotFoundError:
    # Si distutils no existe en este Python, creamos un módulo virtual en memoria
    module_distutils = types.ModuleType("distutils")
    module_version = types.ModuleType("distutils.version")
    
    # Creamos un reemplazo simple de LooseVersion compatible con lo que usa el driver
    class LooseVersion:
        def __init__(self, vstring):
            self.vstring = vstring
            self.version = [int(x) if x.isdigit() else x for x in re.split(r'[^a-zA-Z0-9]', vstring)]
        def __lt__(self, other):
            other_version = other.version if isinstance(other, LooseVersion) else [int(x) if x.isdigit() else x for x in re.split(r'[^a-zA-Z0-9]', str(other))]
            return self.version < other_version
        def __ge__(self, other):
            other_version = other.version if isinstance(other, LooseVersion) else [int(x) if x.isdigit() else x for x in re.split(r'[^a-zA-Z0-9]', str(other))]
            return self.version >= other_version
        def __str__(self):
            return self.vstring
            
    module_version.LooseVersion = LooseVersion
    module_distutils.version = module_version
    sys.modules["distutils"] = module_distutils
    sys.modules["distutils.version"] = module_version

# --- IMPORTACIONES NORMALES ---
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
from selenium.common.exceptions import WebDriverException, TimeoutException

def scrape_restaurantes_info(url):
    if not url:
        print("Error: URL vacia o no localizada para el raspado.")
        return None
    
    try:
        options = uc.ChromeOptions()
        # options.add_argument("--headless") # Lo dejamos comentado para que puedas verlo
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
        
        print("Iniciando Chrome indetectable (esto puede tardar unos segundos)...")
                # Forzamos a que use la versión 151 de tu Google Chrome
        driver = uc.Chrome(options=options, version_main=151)

        driver.set_page_load_timeout(30)
        
    except Exception as e:
        print(f"Error al inicializar el driver indetectable: {e}")
        return None

    try:
        print(f"Tratando cargar la página con Selenium Indetectable: {url}")
        driver.get(url)
        
        # --- PAUSA MANUAL EN JUPYTER ---
        print("\n⚠️ ¡ATENCIÓN!")
        print("1. Si la página carga directamente los restaurantes, presiona ENTER abajo.")
        print("2. Si aparece un CAPTCHA, resuélvelo en la ventana de Chrome y luego presiona ENTER abajo.")
        input("Presiona ENTER aquí para guardar el HTML y continuar...")
        
        response_text = driver.page_source
    except TimeoutException:
        print(f"Error de limite de tiempo al cargar la página ({url}).")
        return None
    except WebDriverException as e:
        print(f"Error al cargar la página {url}: {e}.")
        return None
    finally:
        driver.quit()
        
    soup = BeautifulSoup(response_text, 'html.parser')
    return soup    

# --- Código de ejecución ---
soup_tripadvisor = None
target_url = globals().get('tripadvisor_url') or locals().get('tripadvisor_url')

if target_url:
    print(f"\nTratando raspar la página identificada: {target_url}")
    soup_tripadvisor = scrape_restaurantes_info(target_url)

    if soup_tripadvisor:
        print("HTML de la página de Tripadvisor obtenido con éxito!")
        page_title_tag = soup_tripadvisor.find('title')
        if page_title_tag:
            print(f"Título de la página obtenido: {page_title_tag.get_text(strip=True)}")
        else:
            print("No fue posible encontrar el título de la página.")
    else:
        print("Falla al raspar la página de Tripadvisor.")
else:
    print("No existe URL de Tripadvisor válida para el raspado.")

print("*" * 50)



Tratando raspar la página identificada: https://www.tripadvisor.com.mx/Restaurants-g150813-Tulum_Yucatan_Peninsula.html
Iniciando Chrome indetectable (esto puede tardar unos segundos)...
Tratando cargar la página con Selenium Indetectable: https://www.tripadvisor.com.mx/Restaurants-g150813-Tulum_Yucatan_Peninsula.html

⚠️ ¡ATENCIÓN!
1. Si la página carga directamente los restaurantes, presiona ENTER abajo.
2. Si aparece un CAPTCHA, resuélvelo en la ventana de Chrome y luego presiona ENTER abajo.
HTML de la página de Tripadvisor obtenido con éxito!
Título de la página obtenido: LOS 10 MEJORES restaurantes en Tulum (2026) - Tripadvisor
**************************************************


In [71]:
from bs4 import BeautifulSoup
import re

# Usar el HTML real obtenido de Selenium
if 'soup_tripadvisor' in globals() and soup_tripadvisor:
    soup_completo = soup_tripadvisor
else:
    print("ADVERTENCIA: No se encontró 'soup_tripadvisor' en memoria.")
    soup_completo = None

if soup_completo:
    # Encontrar todos los bloques de restaurante
    restaurant_blocks = soup_completo.find_all('div', {'data-automation': 'restaurantCard'})

    print("Iniciando la extracción detallada de cada restaurante:")
    restaurantes_detallados = []

    # Función de ayuda para buscar clases ignorando mayúsculas y minúsculas
    def check_class(c):
        if not c:
            return False
        if isinstance(c, list):
            c = " ".join(c)
        return "znjnf" in c.lower()

    for block in restaurant_blocks:
        # 1. Nombre del Restaurante y URL
        nombre_link_tag = None
        links = block.find_all('a', href=lambda h: h and h.startswith('/Restaurant_Review-'))
        for l in links:
            text = l.get_text(strip=True)
            if text:  # Si el enlace contiene texto, es el del título/nombre
                nombre_link_tag = l
                break
                
        nombre = "Nombre no encontrado"
        url_restaurante = "URL no encontrada"
        if nombre_link_tag:
            nombre = nombre_link_tag.get_text(strip=True)
            # Limpieza: quitamos el número del principio si existe (ej. "1. Alma Verde" -> "Alma Verde")
            nombre = re.sub(r'^\d+\.\s*', '', nombre)
            url_restaurante = "https://www.tripadvisor.com.mx" + nombre_link_tag['href']

        # 2. Calificacion (ej., 4.9)
        calificacion_tag = block.find('div', {'data-automation': 'bubbleRatingValue'})
        calificacion = calificacion_tag.find('span').get_text(strip=True) if calificacion_tag else "Calificación no encontrada"

        # 3. Cantidad de Reseñas (Búsqueda por palabra clave)
        reviews_span = block.find('span', string=re.compile(r'opiniones|reseñas|reviews', re.IGNORECASE))
        cantidad_resenas = "Cantidad de reseñas no encontrada"
        if reviews_span:
            text_content = reviews_span.get_text(strip=True)
            match = re.search(r'[\d\.,]+', text_content)
            if match:
                cantidad_resenas = match.group(0).replace('.', '').replace(',', '')

        # 4. Tipo de Cocina y 5. Rango de Precios (Búsqueda inteligente por contenido)
        tipo_cocina = []
        rango_precio = "Rango de precios no encontrado"
        spans = block.find_all('span', class_=check_class)
        for span in spans:
            text = span.get_text(strip=True)
            if not text:
                continue
            if '$' in text:
                rango_precio = text
            # --- CORRECCIÓN: Filtramos opiniones y textos con paréntesis de la cocina ---
            elif any(word in text.lower() for word in ['abierto', 'cerrado', 'menú', 'menu', 'patrocinado', 'opiniones', 'opinión', 'opinion', 'reseñas', 'reseña', 'reviews', 'review']) or (text.startswith('(') and text.endswith(')')):
                continue
            else:
                tipo_cocina.append(text)
        
        tipo_cocina_str = tipo_cocina[0] if tipo_cocina else "Tipo de cocina no encontrado"
        tipo_cocina_str = tipo_cocina_str.split(',')[0].strip() if tipo_cocina_str else tipo_cocina_str

        # 6. Estado de Funcionamiento (Búsqueda por palabra clave)
        status_span = block.find('span', string=re.compile(r'abierto|cerrado|abre en|abierto ahora', re.IGNORECASE))
        status_funcionamiento = status_span.get_text(strip=True) if status_span else "Estado de funcionamiento no encontrado"

        # 8. URL de la Imagen Principal (Búsqueda directa de la etiqueta img)
        image_tag = block.find('img', src=re.compile(r'https?://.*(photo|dynamic-media|media).*'))
        if not image_tag:
            image_tag = block.find('img', src=lambda s: s and s.startswith('http'))
        url_imagen = image_tag['src'] if image_tag else "URL de la imagen no encontrada"
        
        # Almacenar los datos con el formato de llaves exactas del tutorial
        restaurante_info = {
            "Nombre": nombre,
            "Calificacion": calificacion,
            "Cantidad_Resenas": cantidad_resenas,
            "Tipo_Cocina": tipo_cocina_str,
            "Rango_Precio": rango_precio,
            "Estado_Funcionamiento": status_funcionamiento,
            "URL_Restaurante": url_restaurante,
            "URL_Imagen_Principal": url_imagen
        }
        restaurantes_detallados.append(restaurante_info)

    # Imprimir los resultados con el formato visual exacto del tutorial
    for i, restaurante in enumerate(restaurantes_detallados):
        print(f"\n--- Restaurante {i+1} ---")
        for key, value in restaurante.items():
            print(f"{key}: {value}")


Iniciando la extracción detallada de cada restaurante:

--- Restaurante 1 ---
Nombre: La Taquería - Pinches Tacos Shop
Calificacion: 4.7
Cantidad_Resenas: 992
Tipo_Cocina: Bar
Rango_Precio: $$ - $$$
Estado_Funcionamiento: Abierto ahora
URL_Restaurante: https://www.tripadvisor.com.mx/Restaurant_Review-g150813-d17729429-Reviews-La_Taqueria_Pinches_Tacos_Shop-Tulum_Yucatan_Peninsula.html
URL_Imagen_Principal: https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/6d/d2/9d/taquiza-deliciosa.jpg?w=200&h=200&s=1

--- Restaurante 2 ---
Nombre: Alma Verde Tulum
Calificacion: 4.9
Cantidad_Resenas: 365
Tipo_Cocina: Mexicana
Rango_Precio: $$ - $$$
Estado_Funcionamiento: Abierto ahora
URL_Restaurante: https://www.tripadvisor.com.mx/Restaurant_Review-g150813-d24855625-Reviews-Alma_Verde_Tulum-Tulum_Yucatan_Peninsula.html
URL_Imagen_Principal: https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/ef/ff/c4/caption.jpg?w=200&h=200&s=1

--- Restaurante 3 ---
Nombre: La Negra Tomasa
Calificacio

In [72]:
from bs4 import BeautifulSoup
import re

# Evitamos borrar lo que guardó Selenium en memoria
if 'soup_tripadvisor' not in globals() or not soup_tripadvisor:
    html_do_tripadvisor = """
    <!DOCTYPE html><html lang="en-US"><head><link rel="icon" id="favicon" type="image/x-icon" href="https://static.tacdn.com/favicon.ico?v2"><meta name="description" content="Tulum Mid Range Restaurants: See 38,727 Tripadvisor traveler reviews of Mid Range Restaurants in Tulum."><meta name="viewport" content="width=device-width, initial-scale=1.0"><link rel="canonical" href="https://www.tripadvisor.com/Restaurants-g150813-zfp10955-Tulum_Yucatan_Peninsula.html"><meta property="og:url" content="https://www.tripadvisor.com/Restaurants-g150813-zfp10955-Tulum_Yucatan_Peninsula.html">
    """
    soup_tripadvisor = BeautifulSoup(html_do_tripadvisor, 'html.parser')

print("\nIniciando la extracción detallada de los restaurantes de la página de Tripadvisor...")
restaurantes_detallados = []

restaurant_blocks = soup_tripadvisor.find_all('div', {'data-automation': 'restaurantCard'})

if not restaurant_blocks:
    print("AVISO: No se encontró ningún bloque principal de restaurantes con los selectores configurados (('data-automation': 'restaurantCard')).")

# Función de ayuda para buscar clases ignorando mayúsculas y minúsculas
def check_class(c):
    if not c:
        return False
    if isinstance(c, list):
        c = " ".join(c)
    return "znjnf" in c.lower()

top_n_restaurants = restaurant_blocks[:5]

for block in top_n_restaurants:
    # 1. Nombre y Enlace (Búsqueda del enlace que sí contiene texto)
    nombre_link_tag = None
    links = block.find_all('a', href=lambda h: h and h.startswith('/Restaurant_Review-'))
    for l in links:
        text = l.get_text(strip=True)
        if text:  # Si el enlace contiene texto, es el del título/nombre
            nombre_link_tag = l
            break
            
    nombre = "Nombre no encontrado"
    link = "Enlace no encontrado"
    if nombre_link_tag:
        nombre = nombre_link_tag.get_text(strip=True)
        nombre = re.sub(r'^\d+\.\s*', '', nombre) # Quita el número del principio (ej. "1. ")
        link = "https://www.tripadvisor.com.mx" + nombre_link_tag['href']
            
    # 2. Reseñas (Búsqueda por palabra clave)
    reviews_span = block.find('span', string=re.compile(r'opiniones|reseñas|reviews', re.IGNORECASE))
    reviews = "Reseñas no encontradas"
    if reviews_span:
        reviews = reviews_span.get_text(strip=True)

    # 3. Calificación (data-automation)
    rating_tag = block.find('div', {'data-automation': 'bubbleRatingValue'})
    rating = rating_tag.find('span').get_text(strip=True) if rating_tag and rating_tag.find('span') else "Calificación no encontrada"

    # 4. Tipo de cocina y 5. Precio (Búsqueda inteligente por contenido)
    tipo_culinaria = "Tipo de cocina no encontrado"
    precio = "Precio no encontrado"
    
    spans = block.find_all('span', class_=check_class)
    for span in spans:
        text = span.get_text(strip=True)
        if not text:
            continue
        if '$' in text:
            precio = text
        elif any(word in text.lower() for word in ['abierto', 'cerrado', 'menú', 'menu', 'patrocinado']):
            continue
        else:
            tipo_culinaria = text

    # 6. Ubicación (desde el nombre)
    localizacion = "Ubicación no especificada (desde el nombre)"
    if ' - ' in nombre:
        partes_nombre = nombre.split(' - ')
        if len(partes_nombre) > 1:
            localizacion = partes_nombre[-1].strip()

    restaurantes_detallados.append({
        "Nombre": nombre,
        "Calificación": rating,
        "Reseñas": reviews,
        "Precio": precio,
        "Tipo de cocina": tipo_culinaria,
        "Ubicacion": localizacion,
        "Enlace": link
    })

if restaurantes_detallados:
    print(f"\n--- {len(restaurantes_detallados)} Restaurantes Extraidos de Tripadvisor ---")
    for i, r in enumerate(restaurantes_detallados):
        print(f"Restaurante #{i+1}:")
        print(f"  Nombre: {r['Nombre']}")
        print(f"  Calificación: {r['Calificación']}")
        print(f"  Reseñas: {r['Reseñas']}")
        print(f"  Precio: {r['Precio']}")
        print(f"  Tipo de Cocina: {r['Tipo de cocina']}")
        print(f"  Ubicacion: {r['Ubicacion']}")
        print(f"  Enlace: {r['Enlace']}")
        print("*" * 40)
else:
    print("No se extrajo ningún detalle de restaurantes. **Verifique los selectores HTML**.")

print("*" * 50)



Iniciando la extracción detallada de los restaurantes de la página de Tripadvisor...

--- 5 Restaurantes Extraidos de Tripadvisor ---
Restaurante #1:
  Nombre: La Taquería - Pinches Tacos Shop
  Calificación: 4.7
  Reseñas: (992 opiniones)
  Precio: $$ - $$$
  Tipo de Cocina: Bar, Restaurantes cerveceros
  Ubicacion: Pinches Tacos Shop
  Enlace: https://www.tripadvisor.com.mx/Restaurant_Review-g150813-d17729429-Reviews-La_Taqueria_Pinches_Tacos_Shop-Tulum_Yucatan_Peninsula.html
****************************************
Restaurante #2:
  Nombre: Alma Verde Tulum
  Calificación: 4.9
  Reseñas: (365 opiniones)
  Precio: $$ - $$$
  Tipo de Cocina: Mexicana, Café
  Ubicacion: Ubicación no especificada (desde el nombre)
  Enlace: https://www.tripadvisor.com.mx/Restaurant_Review-g150813-d24855625-Reviews-Alma_Verde_Tulum-Tulum_Yucatan_Peninsula.html
****************************************
Restaurante #3:
  Nombre: La Negra Tomasa
  Calificación: 4.9
  Reseñas: (1,933 opiniones)
  Precio: $$ 